<h2>Exercise 02. Join</h2>

In [22]:
import pandas as pd
import sqlite3

### 1. Create a connection to the database using the sqlite3 library.

In [23]:
conn = sqlite3.connect("../data/checking-logs.sqlite")

### 2. Create a new table called datamart in the database by joining the tables pageviews and checker using only one query.

In [24]:
datamart = pd.io.sql.read_sql("""
SELECT
    c.uid,
    c.labname,
    c.timestamp AS first_commit_ts,
    pv.first_view_ts
FROM checker c
LEFT JOIN (
    SELECT uid, MIN(datetime) AS first_view_ts
    FROM pageviews
    GROUP BY uid
) pv
ON c.uid = pv.uid
WHERE c.status = 'ready'
AND c.numTrials = 1
AND c.labname IN ('laba04','laba04s','laba05','laba06','laba06s','project1')
AND c.uid LIKE 'user_%';
""",
conn,
parse_dates = ['first_commit_ts', 'first_view_ts'])

datamart.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    str           
 1   labname          140 non-null    str           
 2   first_commit_ts  140 non-null    datetime64[us]
 3   first_view_ts    59 non-null     datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 4.5 KB


### 3. Using Pandas methods, create two dataframes: test and control.

In [25]:
datamart

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,NaT
1,user_4,laba04,2020-04-17 11:33:17.366400,NaT
2,user_4,laba04s,2020-04-17 11:48:41.992466,NaT
3,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
4,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
...,...,...,...,...
135,user_23,laba06,2020-05-21 08:34:10.517205,NaT
136,user_19,laba06s,2020-05-21 13:27:06.705881,2020-04-21 20:30:38.034966
137,user_23,laba06s,2020-05-21 14:29:15.709568,NaT
138,user_17,laba06,2020-05-21 15:21:31.567615,2020-04-18 10:56:55.833899


In [26]:
test = datamart[datamart["first_view_ts"].notna()].copy()
control = datamart[datamart["first_view_ts"].isna()].copy()

In [27]:
test

,uid,labname,first_commit_ts,first_view_ts
3,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
4,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
7,user_30,laba04s,2020-04-18 14:51:37.498399,2020-04-17 22:46:26.785035
8,user_14,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447
11,user_14,laba04s,2020-04-18 22:30:30.247628,2020-04-18 10:53:52.623447
18,user_19,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966
19,user_25,laba04,2020-04-20 19:16:50.673054,2020-05-09 23:54:54.260791
20,user_21,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081
21,user_30,project1,2020-04-22 12:36:24.053518,2020-04-17 22:46:26.785035
23,user_21,laba04s,2020-04-22 20:09:21.857747,2020-04-22 22:40:36.824081


In [28]:
control

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,NaT
1,user_4,laba04,2020-04-17 11:33:17.366400,NaT
2,user_4,laba04s,2020-04-17 11:48:41.992466,NaT
5,user_2,laba04,2020-04-18 13:42:35.482008,NaT
6,user_2,laba04s,2020-04-18 13:51:22.291271,NaT
...,...,...,...,...
126,user_2,laba06s,2020-05-19 14:45:03.908268,NaT
132,user_6,laba06s,2020-05-20 14:50:07.609937,NaT
134,user_7,laba06s,2020-05-20 23:05:37.742597,NaT
135,user_23,laba06,2020-05-21 08:34:10.517205,NaT


In [29]:
mean_ts = test['first_view_ts'].mean()
control.fillna(mean_ts, inplace=True)

mean_ts


Timestamp('2020-04-27 00:40:05.761783')

In [30]:
control["first_view_ts"] = control["first_view_ts"].fillna(mean_ts)
control

,uid,labname,first_commit_ts,first_view_ts
0,user_4,project1,2020-04-17 05:19:02.744528,2020-04-27 00:40:05.761783
1,user_4,laba04,2020-04-17 11:33:17.366400,2020-04-27 00:40:05.761783
2,user_4,laba04s,2020-04-17 11:48:41.992466,2020-04-27 00:40:05.761783
5,user_2,laba04,2020-04-18 13:42:35.482008,2020-04-27 00:40:05.761783
6,user_2,laba04s,2020-04-18 13:51:22.291271,2020-04-27 00:40:05.761783
...,...,...,...,...
126,user_2,laba06s,2020-05-19 14:45:03.908268,2020-04-27 00:40:05.761783
132,user_6,laba06s,2020-05-20 14:50:07.609937,2020-04-27 00:40:05.761783
134,user_7,laba06s,2020-05-20 23:05:37.742597,2020-04-27 00:40:05.761783
135,user_23,laba06,2020-05-21 08:34:10.517205,2020-04-27 00:40:05.761783


In [31]:
test.to_sql("test", conn, if_exists="replace", index=False)
control.to_sql("control", conn, if_exists="replace", index=False)

81

In [32]:
control.info()

<class 'pandas.DataFrame'>
Index: 81 entries, 0 to 137
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     str           
 1   labname          81 non-null     str           
 2   first_commit_ts  81 non-null     datetime64[us]
 3   first_view_ts    81 non-null     datetime64[us]
dtypes: datetime64[us](2), str(2)
memory usage: 3.2 KB


In [33]:
conn.close()